# 🤖 : Parsing Natural Language into Robotic Actions Using LLMs


You'll learn how to use Large Language Models (LLMs) to convert everyday language like **"pick up the mug"** into structured robotic commands that a robot can understand and execute.

### What You'll Learn:
1. **What is Natural Language Parsing?** - Understanding how robots can understand human language
2. **Grammar for Robotic Actions** - Creating a fixed structure for robot commands
3. **Using LLMs as Parsers** - Teaching an AI to translate language into robot actions
4. **Validation & Safety** - Making sure commands are safe before execution
5. **Testing with 20+ Instructions** - Trying diverse real-world examples


### Key Concepts:
- **LLM (Large Language Model)**: AI that understands and generates text (like ChatGPT)
- **Parser**: A program that breaks down text into structured data
- **Grammar**: A set of rules that define valid robot commands
- **Action**: A single thing a robot can do (pick, place, move, etc.)

Let's get started! 🚀

## Step 1: Installation and Setup

First, we need to install the required libraries. We'll use:
- **guidance**: A library that helps control LLM outputs with grammar rules
- **openai**: To connect to GPT models (or we can use open-source alternatives)
- **pydantic**: For data validation (making sure our robot commands are correct)

Run the cell below to install everything:

In [1]:
!pip install -U ipykernel

In [2]:
# Checking if .env file exists and if it does, it shows the file size and the first 50 characters of the file
import os

# Check if file exists
if os.path.exists('.env'):
    print("✅ .env file found!")
    
    # Show file size
    size = os.path.getsize('.env')
    print(f"   File size: {size} bytes")
    
    # Show content (first 50 chars for privacy)
    with open('.env', 'r') as f:
        content = f.read().strip()
        print(f"   Content starts with: {content[:50]}...")
else:
    print("❌ .env file not found in current directory")
    print(f"   Current directory: {os.getcwd()}")

✅ .env file found!
   File size: 181 bytes
   Content starts with: OPENAI_API_KEY=sk-proj-pl_aQqz-Z_9Dh6RN8x3J-GWhyCE...


In [5]:
# Install required packages

!pip install guidance openai pydantic python-dotenv -q

print("✅ All packages installed successfully!")

✅ All packages installed successfully!


In [6]:
from dotenv import load_dotenv
load_dotenv()  # This loads the .env file

print("✅ Environment loaded!")

✅ Environment loaded!


## Step 2: Understanding Robotic Actions

Before we dive into code, let's understand what robotic actions look like.

### Human Language vs Robot Language:

| Human Says | Robot Needs |
|------------|-------------|
| "Pick up the mug" | `(pick mug)` |
| "Place the mug near the laptop" | `(place mug near laptop)` |
| "Move to the kitchen" | `(move_to kitchen)` |
| "Open the door" | `(open door)` |

### Why Does This Matter?
Robots are like very literal friends - they need **exact, structured instructions**. Our job is to translate natural, flexible human language into this precise format.

### Our Grammar Rules:
We'll define a **grammar** (set of rules) for robot actions. Think of it like sentence structure in English class, but for robots!

**Valid Action Types:**
- `pick <object>` - Pick up an object
- `place <object> <location>` - Put an object somewhere
- `move_to <location>` - Move robot to a location
- `open <object>` - Open something (door, drawer)
- `close <object>` - Close something
- `push <object>` - Push an object
- `pull <object>` - Pull an object

## Step 3: Define the Robot Action Grammar

Now let's write code to define what valid robot actions look like. We'll use Python classes to represent different types of actions.



In [7]:
from pydantic import BaseModel, Field, model_validator
from typing import List, Optional, Literal
from enum import Enum


In [11]:

# ========================================
# DEFINING WHAT ACTIONS A ROBOT CAN DO
# ========================================

class ActionType(str, Enum):
    """
    This defines ALL the types of actions our robot can perform.
    
    An Enum (enumeration) is like a multiple-choice list - the robot can only
    do actions from this list, nothing else!
    This is not a regular class—it's an Enum (Enumeration). An Enum is a special type of class that represents a fixed set of named constants.
    
    Why is this important? 
    - Safety: Prevents the robot from doing undefined/dangerous actions
    - Clarity: We know exactly what the robot CAN do
    """
    PICK = "pick"              # Pick up an object
    PLACE = "place"            # Put down an object
    MOVE_TO = "move_to"        # Move to a location
    OPEN = "open"              # Open something
    CLOSE = "close"            # Close something
    PUSH = "push"              # Push an object
    PULL = "pull"              # Pull an object
    GRASP = "grasp"            # Hold/grasp an object
    RELEASE = "release"        # Let go of an object

In [12]:
class RobotAction(BaseModel):
    """
    This is the MAIN blueprint for a robot action.
    
    Every robot action has:
    - action: What to do (from ActionType above)
    - target: What object to interact with (like "mug", "door")
    - location: Where to do it (optional - only for place/move actions)
    - relation: How objects relate (like "near", "on", "under") - optional
    
    BaseModel comes from pydantic - it automatically checks if data is valid!
    """
    
    # The type of action (must be one from ActionType)
    # Field() is a function from Pydantic that lets you add metadata and 
    # validation rules to class attributes. Think of it as adding "instructions" 
    # or "constraints" to each variable. here we are adding validation for each of class attributes
    # this is also defining the attributes of the class , so your RobotAction object will have the attributes action, target, location, relation and they will be of the type ActionType, str, Optional[str], Optional[str] 
    #the validation of the attributes will be described as follows:
    action: ActionType = Field(
        description="The type of action the robot should perform"
    )
    
    # The object being acted upon (e.g., "mug", "door", "book")
    target: str = Field(
        description="The object or target of the action",
        min_length=1  # Must have at least 1 character
    )
    
    # Where to do the action (optional - only needed for some actions)
    location: Optional[str] = Field(
        None,  # None means it's optional
        description="Location or reference object (for place/move actions)"
    )
    
    # Spatial relationship (near, on, under, etc.)
    relation: Optional[str] = Field(
        None,
        description="Spatial relation like 'near', 'on', 'under', 'next to'"
    )
    
    # ========================================
    # VALIDATION: Making sure the action makes sense!
    # ========================================
    
    # NEW in Pydantic v2: @model_validator
    # This is the BEST way to validate across multiple fields!
    @model_validator(mode='after')
    def validate_location_for_action(self):
        """
        This function automatically checks if the entire action makes sense.
        
        Rule: PLACE and MOVE_TO actions NEED a location.
        Why? You can't place something "nowhere"!
        
        How it works
        - @model_validator checks the WHOLE model after all fields are set
        - mode='after' means this runs AFTER individual field validation
        - 'self' gives us access to ALL fields (self.action, self.location, etc.)
               
        This is called automatically when you create a RobotAction object.
        """
        # Check if action is PLACE or MOVE_TO
        if self.action in [ActionType.PLACE, ActionType.MOVE_TO]:
            # If so, location MUST be provided (not None or empty string)
            if not self.location:
                raise ValueError(
                    f"❌ {self.action.value} action requires a location! "
                    f"You can't {self.action.value} something 'nowhere'. "
                    f"Example: place mug [location: table]"
                )
        
        # If validation passes, return self (the entire object)
        return self
    
    def to_robot_command(self) -> str:
        """
        Convert this action into a simple robot command string.
        
        This is the final format the robot will see!
        Examples:
        - (pick mug)
        - (place mug near laptop)
        - (move_to kitchen)
        
        Returns:
            str: A robot-readable command in parentheses
        """
        # Build the command step by step
        cmd_parts = [self.action.value, self.target]
        
        # Add relation if it exists (like "near", "on")
        if self.relation:
            cmd_parts.append(self.relation)
        
        # Add location if it exists
        if self.location:
            cmd_parts.append(self.location)
        
        # Join all parts with spaces and wrap in parentheses
        return f"({' '.join(cmd_parts)})"


### 📝 Understanding Model Validation (Pydantic v2)


#### What's a Model Validator?

A **model validator** checks if the entire object makes sense AFTER all individual fields are validated. It's perfect for rules that depend on multiple fields!


** Syntax (Pydantic v2 - Modern & Simple):**
```python
@model_validator(mode='after')
def validate_location_for_action(self):
    if self.action == ActionType.PLACE and not self.location:
        raise ValueError("Need location!")
    return self
```
**Benefits:** Clear, uses regular `self`, easier to read!

#### Why is This Better for Beginners?

1. **Intuitive**: Uses `self` just like regular class methods - no special parameters!
2. **Clear Access**: `self.action` and `self.location` are obvious and straightforward
3. **Whole Object**: You can check relationships between ANY fields, not just one
4. **Simpler Logic**: Easier to write "if this AND that" validation rules

#### How It Works Step-by-Step:

```python
# When you create a RobotAction:
action = RobotAction(action=ActionType.PLACE, target="mug", location=None)

# Pydantic does this automatically:
# Step 1: Validate individual fields
#   ✅ action is ActionType? Yes
#   ✅ target has min_length=1? Yes
#   ✅ location is Optional[str]? Yes (None is OK for Optional)

# Step 2: Run model validator (mode='after')
#   ❌ Check: Does PLACE action have location?
#   ❌ No! Raise error: "place action requires a location!"

# Result: You get a clear error BEFORE the robot tries to execute!
```

#### Real Examples:

**✅ Valid - Pick action (no location needed):**
```python
action = RobotAction(
    action=ActionType.PICK,
    target="mug"
)
# Works! Pick doesn't need a location
```

**✅ Valid - Place action (with location):**
```python
action = RobotAction(
    action=ActionType.PLACE,
    target="mug",
    location="table"
)
# Works! Has required location
```

**❌ Invalid - Place action (missing location):**
```python
action = RobotAction(
    action=ActionType.PLACE,
    target="mug"
    # location is None by default
)
# Error: "❌ place action requires a location!"
```

#### Why This Matters for Robotics:

In robotics, **validation prevents disasters**:
- Telling a robot to place something with no location → Robot doesn't know where to go!
- Catching errors in code → Much better than discovering them when robot is running
- Clear error messages → Easy to debug and fix

**Safety First:** Always validate before sending commands to hardware! 🤖🛡️

In [13]:
# ========================================
# QUICK TEST: Let's verify the validator works!
# ========================================

print("🧪 Testing the Model Validator\n")
print("=" * 60)

# Test 1: Valid pick action (no location needed)
print("\n✅ Test 1: Pick action without location")
try:
    test1 = RobotAction(action=ActionType.PICK, target="mug")
    print(f"   Success! Created: {test1.action.value} {test1.target}")
except Exception as e:
    print(f"   Error: {e}")

# Test 2: Valid place action (with location)
print("\n✅ Test 2: Place action with location")
try:
    test2 = RobotAction(
        action=ActionType.PLACE,
        target="mug",
        location="table"
    )
    print(f"   Success! Created: {test2.action.value} {test2.target} at {test2.location}")
except Exception as e:
    print(f"   Error: {e}")

# Test 3: Invalid place action (missing location) - Should fail!
print("\n❌ Test 3: Place action WITHOUT location (should fail)")
try:
    test3 = RobotAction(action=ActionType.PLACE, target="mug")
    print(f"   Unexpected: This should have failed!")
except Exception as e:
    print(f"   ✅ Correctly caught error!")
    print(f"   Error message: {str(e).split('Value error,')[1].split('[')[0].strip()}")

print("\n" + "=" * 60)
print("🎉 Validator is working correctly!")

🧪 Testing the Model Validator


✅ Test 1: Pick action without location
   Success! Created: pick mug

✅ Test 2: Place action with location
   Success! Created: place mug at table

❌ Test 3: Place action WITHOUT location (should fail)
   ✅ Correctly caught error!
   Error message: ❌ place action requires a location! You can't place something 'nowhere'. Example: place mug

🎉 Validator is working correctly!


In [14]:
class RobotPlan(BaseModel):
    """
    A PLAN is a sequence (list) of actions.
    
    Why do we need plans?
    Because complex tasks need multiple steps!
    
    Example: "Bring me the mug" needs:
    1. (move_to table)
    2. (pick mug)
    3. (move_to person)
    4. (place mug near person)
    """
    
    # The original instruction in human language
    instruction: str = Field(
        description="The original natural language instruction"
    )
    
    # List of actions that make up this plan
    actions: List[RobotAction] = Field(
        description="Sequence of robot actions to execute",
        min_items=1  # Must have at least 1 action!
    )
    
    def to_robot_commands(self) -> List[str]:
        """
        Convert all actions in this plan to robot commands.
        
        Returns:
            List[str]: A list of robot commands, one for each action
        """
        return [action.to_robot_command() for action in self.actions]
    
    def validate_plan(self) -> tuple[bool, str]:
        """
        Check if this plan makes logical sense.
        
        This is SUPER important for safety!
        We check for common mistakes like:
        - Trying to place something before picking it up
        - Trying to open something you haven't moved to
        
        Returns:
            tuple: (is_valid, message)
                - is_valid: True if plan is safe, False otherwise
                - message: Explanation of what's wrong (or "Valid!")
        """
        # Keep track of what objects the robot is holding
        held_objects = set()  # A set is like a bag - no duplicates
        
        # Check each action one by one
        for i, action in enumerate(self.actions):
            
            # Rule 1: Can't place something you're not holding!
            if action.action == ActionType.PLACE:
                if action.target not in held_objects:
                    return False, (
                        f"❌ Error at step {i+1}: Trying to place '{action.target}' "
                        f"but robot isn't holding it! Must pick it up first."
                    )
                # After placing, remove from held objects
                held_objects.remove(action.target)
            
            # Rule 2: When you pick something, you're now holding it
            elif action.action == ActionType.PICK:
                held_objects.add(action.target)
            
            # Rule 3: Can't hold too many things at once
            # (Most robots have 1 or 2 grippers)
            if len(held_objects) > 2:
                return False, (
                    f"❌ Error at step {i+1}: Robot is trying to hold "
                    f"{len(held_objects)} objects at once! Maximum is 2."
                )
        
        # If we made it here, the plan is valid!
        return True, "✅ Plan is valid and safe to execute!"



/tmp/ipykernel_662/1161686470.py:21: PydanticDeprecatedSince20: `min_items` is deprecated and will be removed, use `min_length` instead. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.12/migration/
  actions: List[RobotAction] = Field(


In [13]:



print("✅ Robot action grammar defined successfully!")
print("\nAvailable actions:")
for action in ActionType:
    print(f"  - {action.value}")

✅ Robot action grammar defined successfully!

Available actions:
  - pick
  - place
  - move_to
  - open
  - close
  - push
  - pull
  - grasp
  - release


## Step 4: Test the Grammar (Manual Example)

Let's test our grammar by manually creating some robot actions. This helps us understand the structure before we use the LLM.

In [15]:
# ========================================
# TESTING: Creating actions manually
# ========================================

print("🧪 Testing Robot Action Grammar\n")
print("=" * 50)

# Example 1: Simple pick action
print("\n📋 Example 1: Pick up a mug")
action1 = RobotAction(
    action=ActionType.PICK,
    target="mug"
)
print(f"Human: Pick up the mug")
print(f"Robot command: {action1.to_robot_command()}")

# Example 2: Place action with location
print("\n📋 Example 2: Place mug near laptop")
action2 = RobotAction(
    action=ActionType.PLACE,
    target="mug",
    relation="near",
    location="laptop"
)
print(f"Human: Place the mug near the laptop")
print(f"Robot command: {action2.to_robot_command()}")

# Example 3: Move to location
print("\n📋 Example 3: Move to kitchen")
action3 = RobotAction(
    action=ActionType.MOVE_TO,
    target="kitchen",
    location="kitchen"  # For move_to, target and location are often the same
)
print(f"Human: Go to the kitchen")
print(f"Robot command: {action3.to_robot_command()}")

# Example 4: Create a full plan
print("\n📋 Example 4: Full plan with multiple actions")
plan = RobotPlan(
    instruction="Pick up the mug and place it near the laptop",
    actions=[
        RobotAction(action=ActionType.PICK, target="mug"),
        RobotAction(
            action=ActionType.PLACE, 
            target="mug", 
            relation="near", 
            location="laptop"
        )
    ]
)

print(f"Human: {plan.instruction}")
print(f"Robot commands:")
for i, cmd in enumerate(plan.to_robot_commands(), 1):
    print(f"  Step {i}: {cmd}")

# Validate the plan
is_valid, message = plan.validate_plan()
print(f"\n{message}")

print("\n" + "=" * 50)
print("✅ Grammar test complete!")

🧪 Testing Robot Action Grammar


📋 Example 1: Pick up a mug
Human: Pick up the mug
Robot command: (pick mug)

📋 Example 2: Place mug near laptop
Human: Place the mug near the laptop
Robot command: (place mug near laptop)

📋 Example 3: Move to kitchen
Human: Go to the kitchen
Robot command: (move_to kitchen kitchen)

📋 Example 4: Full plan with multiple actions
Human: Pick up the mug and place it near the laptop
Robot commands:
  Step 1: (pick mug)
  Step 2: (place mug near laptop)

✅ Plan is valid and safe to execute!

✅ Grammar test complete!


## Step 5: Setting Up the LLM Parser

Now comes the exciting part! We'll use an LLM to automatically parse natural language into robot actions.

### What is Guidance?
The `guidance` library helps us **constrain** LLM outputs. Instead of letting the LLM say whatever it wants, we guide it to produce ONLY valid robot commands that follow our grammar.

### Two Options:
1. **OpenAI GPT Models** (requires API key - costs money but very good)
2. **Open-Source Models** (free, runs locally, but needs more setup)

We'll set up both options so you can choose!

In [16]:
from openai import OpenAI
# Get API key from environment variable
api_key = os.getenv("OPENAI_API_KEY")
if not api_key:
    print("⚠️  No OpenAI API key found!")
    print("   Set it with: export OPENAI_API_KEY='your-key-here'")
    print("   Or we'll use mock responses for demonstration.")
    client = None
else:
    client = OpenAI(api_key=api_key)
    print("✅ OpenAI client initialized!")

✅ OpenAI client initialized!


In [18]:
import os
import json
from typing import List, Dict
import re

# ========================================
# LLM SETUP
# ========================================

class RobotInstructionParser:
    """
    This is our MAIN PARSER class.
    
    It takes human language and converts it to robot actions using an LLM.
    
    Think of it as a translator:
    - Input: Human language ("pick up the mug")
    - Output: Robot commands ((pick mug))
    
    How it works:
    1. We give the LLM examples of good translations
    2. We tell it the grammar rules
    3. We ask it to translate new instructions
    4. We validate the output
    """
    
    def __init__(self, model_name: str = "gpt-3.5-turbo", use_openai: bool = True):
        """
        Initialize the parser.
        
        Args:
            model_name: Which AI model to use
            use_openai: True for OpenAI, False for open-source
        """
        self.model_name = model_name
        self.use_openai = use_openai
        
        # We'll use OpenAI's API directly for simplicity
        # (guidance has some setup complexities for beginners)
        if use_openai:
            try:
                from openai import OpenAI
                
                # Get API key from environment variable
                api_key = os.getenv("OPENAI_API_KEY")
                if not api_key:
                    print("⚠️  No OpenAI API key found!")
                    print("   Set it with: export OPENAI_API_KEY='your-key-here'")
                    print("   Or we'll use mock responses for demonstration.")
                    self.client = None
                else:
                    self.client = OpenAI(api_key=api_key)
                    print("✅ OpenAI client initialized!")
            except ImportError:
                print("⚠️  OpenAI package not available, using mock responses")
                self.client = None
        else:
            self.client = None
        
        # Store successful parses for learning
        self.parse_history: List[Dict] = []
    
    def _create_system_prompt(self) -> str:
        """
        Create the instructions we give to the LLM.
        
        This is SUPER important! The system prompt teaches the LLM:
        - What actions are valid
        - How to format output
        - Examples of good translations
        
        Returns:
            str: The complete instruction prompt for the LLM
        """
        return """
You are a robot instruction parser. Your job is to convert natural language instructions into structured robot commands.

VALID ACTIONS (and only these):
- pick <object>
- place <object> <relation> <location>
- move_to <location>
- open <object>
- close <object>
- push <object>
- pull <object>
- grasp <object>
- release <object>

RELATIONS: near, on, under, next_to, above, below, inside

OUTPUT FORMAT (JSON):
{
  "actions": [
    {
      "action": "pick",
      "target": "mug",
      "location": null,
      "relation": null
    }
  ]
}

EXAMPLES:

Input: "Pick up the red mug"
Output: {
  "actions": [
    {"action": "pick", "target": "red_mug", "location": null, "relation": null}
  ]
}

Input: "Place the book on the table"
Output: {
  "actions": [
    {"action": "place", "target": "book", "location": "table", "relation": "on"}
  ]
}

Input: "Go to the kitchen and open the fridge"
Output: {
  "actions": [
    {"action": "move_to", "target": "kitchen", "location": "kitchen", "relation": null},
    {"action": "open", "target": "fridge", "location": null, "relation": null}
  ]
}

RULES:
1. Use underscores for multi-word objects (red mug → red_mug)
2. Break complex instructions into multiple actions
3. Add move_to actions when location changes are implied
4. Always output valid JSON
5. Only use actions from the valid list
"""
    
    def parse(self, instruction: str) -> RobotPlan:
        """
        Parse a natural language instruction into a robot plan.
        
        This is the MAIN METHOD you'll use!
        
        Args:
            instruction: Natural language like "pick up the mug"
        
        Returns:
            RobotPlan: A validated plan with robot actions
        
        Raises:
            ValueError: If parsing fails or plan is invalid
        """
        print(f"\n🤖 Parsing: '{instruction}'")
        
        # Step 1: Get response from LLM
        if self.client:
            response_text = self._call_openai(instruction)
            print(f"response_text: {response_text}")

        # Step 2: Parse the JSON response
        try:
            response_data = json.loads(response_text)
            print(f"response_data: {response_data}")
        except json.JSONDecodeError as e:
            raise ValueError(f"LLM returned invalid JSON: {e}")
        
        # Step 3: Convert to RobotAction objects
        actions = []
        #response_data.get("actions", []) - Get "actions" list, or empty list if missing
        
        for action_dict in response_data.get("actions", []):
            try:
                action = RobotAction(
                    action=ActionType(action_dict["action"]),
                    target=action_dict["target"],
                    location=action_dict.get("location"),
                    relation=action_dict.get("relation")
                )
                actions.append(action)
            except Exception as e:
                raise ValueError(f"Invalid action format: {e}")
        
        # Step 4: Create and validate the plan
        # Creates a RobotPlan object with:
        # Original instruction
        # List of RobotAction objects we just created
        # The RobotPlan class handles grouping actions together
        plan = RobotPlan(instruction=instruction, actions=actions)
        
        is_valid, message = plan.validate_plan()
        if not is_valid:
            raise ValueError(f"Plan validation failed: {message}")
        
        # Step 5: Store successful parse
        self.parse_history.append({
            "instruction": instruction,
            "plan": plan,
            "commands": plan.to_robot_commands()
        })
        
        print(f"✅ Parsed successfully!")
        return plan
    
    def _call_openai(self, instruction: str) -> str:
        """
        Call OpenAI API to parse the instruction.
        
        This sends the instruction to GPT and gets back structured JSON.
        """
        response = self.client.chat.completions.create(
            model=self.model_name,
            messages=[
                {"role": "system", "content": self._create_system_prompt()},
                {"role": "user", "content": instruction}
            ],
            temperature=0.1,  # Low temperature = more consistent output
            response_format={"type": "json_object"}  # Force JSON output
        )
        
        return response.choices[0].message.content
    
    


print("✅ Robot Instruction Parser class created!")
print("\n💡 Tip: Set your OpenAI API key to use real LLM parsing:")
print("   export OPENAI_API_KEY='your-key-here'")
print("\n   (We'll use mock responses for demonstration if no key is set)")

✅ Robot Instruction Parser class created!

💡 Tip: Set your OpenAI API key to use real LLM parsing:
   export OPENAI_API_KEY='your-key-here'

   (We'll use mock responses for demonstration if no key is set)


## Step 6: Test the Parser with Simple Examples

Let's test our parser with a few simple instructions to make sure everything works!

In [19]:
# ========================================
# TESTING THE PARSER
# ========================================

# Create the parser
# Note: This will use mock responses if no API key is set
parser = RobotInstructionParser(model_name="gpt-3.5-turbo")

print("🧪 Testing Parser with Simple Examples")
print("=" * 60)

# Test 1: Simple pick
print("\n📝 Test 1: Simple pick command")
try:
    plan1 = parser.parse("Pick up the mug")
    print(f"Commands: {plan1.to_robot_commands()}")
except Exception as e:
    print(f"❌ Error: {e}")

# Test 2: Simple place
print("\n📝 Test 2: Place command")
try:
    plan2 = parser.parse("Place the mug near the laptop")
    print(f"Commands: {plan2.to_robot_commands()}")
except Exception as e:
    print(f"❌ Error: {e}")

# Test 3: Move command
print("\n📝 Test 3: Move command")
try:
    plan3 = parser.parse("Go to the kitchen")
    print(f"Commands: {plan3.to_robot_commands()}")
except Exception as e:
    print(f"❌ Error: {e}")

print("\n" + "=" * 60)
print("✅ Simple tests complete!")

✅ OpenAI client initialized!
🧪 Testing Parser with Simple Examples

📝 Test 1: Simple pick command

🤖 Parsing: 'Pick up the mug'
response_text: {
  "actions": [
    {"action": "pick", "target": "mug", "location": null, "relation": null}
  ]
}
response_data: {'actions': [{'action': 'pick', 'target': 'mug', 'location': None, 'relation': None}]}
✅ Parsed successfully!
Commands: ['(pick mug)']

📝 Test 2: Place command

🤖 Parsing: 'Place the mug near the laptop'
response_text: {
  "actions": [
    {"action": "place", "target": "mug", "location": "laptop", "relation": "near"}
  ]
}
response_data: {'actions': [{'action': 'place', 'target': 'mug', 'location': 'laptop', 'relation': 'near'}]}
❌ Error: Plan validation failed: ❌ Error at step 1: Trying to place 'mug' but robot isn't holding it! Must pick it up first.

📝 Test 3: Move command

🤖 Parsing: 'Go to the kitchen'
response_text: {
  "actions": [
    {"action": "move_to", "target": "kitchen", "location": "kitchen", "relation": null}
  ]
}
re

## Step 7: Testing with 20+ Diverse Instructions

Now for the main event! We'll test our parser with at least 20 diverse, real-world instructions.

### Why 20+ instructions?
- Tests different types of actions
- Tests different complexities (simple to complex)
- Tests edge cases and unusual phrasings
- Ensures the parser is robust (works in many situations)

In [24]:
# ========================================
# 20+ DIVERSE TEST INSTRUCTIONS
# ========================================

# This list covers many different scenarios:
# - Simple actions (1 step)
# - Complex actions (multiple steps)
# - Different objects and locations
# - Different spatial relationships
# - Different phrasings of the same action

test_instructions = [
    # ===== SIMPLE PICK ACTIONS =====
    "Pick up the mug",
    "Grab the book",
    "Take the phone",
    "Get the remote control",
    
    # ===== PLACE ACTIONS (different relations) =====
    "Place the mug on the table",
    "Put the book next to the laptop",
    "Set the phone near the charger",
    "Place the cup under the coffee machine",
    "Put the plate above the napkin",
    
    # ===== MOVEMENT ACTIONS =====
    "Go to the kitchen",
    "Move to the living room",
    "Navigate to the bedroom",
    
    # ===== OPEN/CLOSE ACTIONS =====
    "Open the door",
    "Close the drawer",
    "Open the refrigerator",
    
    # ===== PUSH/PULL ACTIONS =====
    "Push the chair",
    "Pull the handle",
    
    # ===== COMPLEX MULTI-STEP ACTIONS =====
    "Pick up the mug and place it near the laptop",
    "Go to the kitchen and open the fridge",
    "Grab the book and put it on the shelf",
    "Take the remote and place it on the couch",
    
    # ===== CHALLENGING/UNUSUAL PHRASINGS =====
    "Could you please pick up the mug?",  # Polite form
    "I need you to move the cup to the table",  # Indirect
    "The book should go on the desk",  # Passive voice
]

print(f"📝 Running {len(test_instructions)} test instructions")
print("=" * 80)

# Track results
successful_parses = 0
failed_parses = 0
results = []

# Test each instruction
for i, instruction in enumerate(test_instructions, 1):
    print(f"\n[{i}/{len(test_instructions)}] Testing: \"{instruction}\"")
    print("-" * 80)
    
    try:
        # Parse the instruction
        plan = parser.parse(instruction)
        
        # Get robot commands
        commands = plan.to_robot_commands()
        
        # Display results
        print(f"✅ Success! Generated {len(commands)} action(s):")
        for j, cmd in enumerate(commands, 1):
            print(f"   {j}. {cmd}")
        
        # Validate
        is_valid, message = plan.validate_plan()
        print(f"   {message}")
        
        successful_parses += 1
        results.append({
            "instruction": instruction,
            "success": True,
            "commands": commands,
            "validation": message
        })
        
    except Exception as e:
        print(f"❌ Failed: {e}")
        failed_parses += 1
        results.append({
            "instruction": instruction,
            "success": False,
            "error": str(e)
        })

# ========================================
# SUMMARY REPORT
# ========================================

print("\n" + "=" * 80)
print("📊 FINAL RESULTS")
print("=" * 80)
print(f"\n✅ Successful parses: {successful_parses}/{len(test_instructions)}")
print(f"❌ Failed parses: {failed_parses}/{len(test_instructions)}")
print(f"📈 Success rate: {(successful_parses/len(test_instructions)*100):.1f}%")

# Show any failures in detail
if failed_parses > 0:
    print("\n⚠️  Failed instructions:")
    for result in results:
        if not result["success"]:
            print(f"   - \"{result['instruction']}\"")
            print(f"     Error: {result['error']}")

print("\n" + "=" * 80)
print("🎉 Testing complete!")

📝 Running 24 test instructions

[1/24] Testing: "Pick up the mug"
--------------------------------------------------------------------------------

🤖 Parsing: 'Pick up the mug'
✅ Parsed successfully!
✅ Success! Generated 1 action(s):
   1. (pick mug)
   ✅ Plan is valid and safe to execute!

[2/24] Testing: "Grab the book"
--------------------------------------------------------------------------------

🤖 Parsing: 'Grab the book'
✅ Parsed successfully!
✅ Success! Generated 1 action(s):
   1. (pick book)
   ✅ Plan is valid and safe to execute!

[3/24] Testing: "Take the phone"
--------------------------------------------------------------------------------

🤖 Parsing: 'Take the phone'
✅ Parsed successfully!
✅ Success! Generated 1 action(s):
   1. (pick phone)
   ✅ Plan is valid and safe to execute!

[4/24] Testing: "Get the remote control"
--------------------------------------------------------------------------------

🤖 Parsing: 'Get the remote control'
✅ Parsed successfully!
✅ Success

## Step 8: Visualizing the Results

Let's create a nice summary table of all our test results!

In [22]:
import pandas as pd
from IPython.display import display, HTML

# ========================================
# CREATE RESULTS TABLE
# ========================================

print("📊 Creating results visualization...\n")

# Convert results to a pandas DataFrame (table)
table_data = []
for i, result in enumerate(results, 1):
    if result["success"]:
        table_data.append({
            "#": i,
            "Instruction": result["instruction"],
            "Status": "✅",
            "Commands": " → ".join(result["commands"]),
            "# Actions": len(result["commands"])
        })
    else:
        table_data.append({
            "#": i,
            "Instruction": result["instruction"],
            "Status": "❌",
            "Commands": f"Error: {result['error'][:50]}...",
            "# Actions": 0
        })

df = pd.DataFrame(table_data)

# Display the table
print("Results Table:")
print("=" * 80)
display(df)

# Statistics
print("\n📈 Statistics:")
print(f"   Average actions per instruction: {df['# Actions'].mean():.2f}")
print(f"   Most complex instruction: {df['# Actions'].max()} actions")
print(f"   Simplest instruction: {df[df['# Actions'] > 0]['# Actions'].min()} action(s)")

📊 Creating results visualization...

Results Table:


,#,Instruction,Status,Commands,# Actions
0,1,Pick up the mug,✅,(pick mug),1
1,2,Grab the book,✅,(pick book),1
2,3,Take the phone,✅,(pick phone),1
3,4,Get the remote control,✅,(pick remote_control),1
4,5,Place the mug on the table,❌,Error: Plan validation failed: ❌ Error at step...,0
5,6,Put the book next to the laptop,❌,Error: Plan validation failed: ❌ Error at step...,0
6,7,Set the phone near the charger,❌,Error: Plan validation failed: ❌ Error at step...,0
7,8,Place the cup under the coffee machine,❌,Error: Plan validation failed: ❌ Error at step...,0
8,9,Put the plate above the napkin,❌,Error: Plan validation failed: ❌ Error at step...,0
9,10,Go to the kitchen,✅,(move_to kitchen kitchen),1



📈 Statistics:
   Average actions per instruction: 1.12
   Most complex instruction: 3 actions
   Simplest instruction: 1 action(s)


## Step 9: Interactive Testing

Now it's YOUR turn! Use this interactive cell to test your own instructions.

In [20]:
# ========================================
# INTERACTIVE TESTING
# ========================================

def test_instruction(instruction: str):
    """
    Test a single instruction and display detailed results.
    
    This is a helper function to make testing easier and prettier!
    """
    print("=" * 80)
    print(f"🤖 Testing: \"{instruction}\"")
    print("=" * 80)
    
    try:
        # Parse
        plan = parser.parse(instruction)
        
        # Show actions
        print("\n📋 Generated Actions:")
        for i, action in enumerate(plan.actions, 1):
            print(f"\n   Step {i}:")
            print(f"   - Action Type: {action.action.value}")
            print(f"   - Target: {action.target}")
            if action.location:
                print(f"   - Location: {action.location}")
            if action.relation:
                print(f"   - Relation: {action.relation}")
        
        # Show robot commands
        print("\n🤖 Robot Commands:")
        for i, cmd in enumerate(plan.to_robot_commands(), 1):
            print(f"   {i}. {cmd}")
        
        # Validate
        is_valid, message = plan.validate_plan()
        print(f"\n{message}")
        
    except Exception as e:
        print(f"\n❌ Error: {e}")
    
    print("\n" + "=" * 80)


# Try your own instruction!
# Change the text below to test different instructions

print("🎮 Interactive Testing Mode\n")
print("Try these examples or write your own!\n")

# Example 1
test_instruction("Pick up the red mug and place it on the kitchen counter")

# Example 2 - Uncomment to try!
# test_instruction("Go to the bedroom, open the closet, and grab the blue shirt")

# Write your own here! Uncomment and modify:
# test_instruction("Your instruction here")

🎮 Interactive Testing Mode

Try these examples or write your own!

🤖 Testing: "Pick up the red mug and place it on the kitchen counter"

🤖 Parsing: 'Pick up the red mug and place it on the kitchen counter'
response_text: {
  "actions": [
    {"action": "pick", "target": "red_mug", "location": null, "relation": null},
    {"action": "move_to", "target": "kitchen", "location": "kitchen", "relation": null},
    {"action": "place", "target": "red_mug", "location": "kitchen_counter", "relation": "on"}
  ]
}
response_data: {'actions': [{'action': 'pick', 'target': 'red_mug', 'location': None, 'relation': None}, {'action': 'move_to', 'target': 'kitchen', 'location': 'kitchen', 'relation': None}, {'action': 'place', 'target': 'red_mug', 'location': 'kitchen_counter', 'relation': 'on'}]}
✅ Parsed successfully!

📋 Generated Actions:

   Step 1:
   - Action Type: pick
   - Target: red_mug

   Step 2:
   - Action Type: move_to
   - Target: kitchen
   - Location: kitchen

   Step 3:
   - Action Typ

## Step 10: Advanced Features - Improving the Parser

Now let's add some advanced features to make our parser even better!

### New Features:
1. **Complexity Analysis** - How complex is the instruction?
2. **Ambiguity Detection** - Flag unclear instructions
3. **Confidence Scoring** - How confident is the parser?

In [ ]:
# ========================================
# ADVANCED PARSER FEATURES
# ========================================

class AdvancedRobotParser(RobotInstructionParser):
    """
    An enhanced version of our parser with extra features.
    
    This inherits from (extends) RobotInstructionParser,
    meaning it has all the original features PLUS new ones!
    """
    
    def analyze_instruction(self, instruction: str) -> Dict:
        """
        Analyze an instruction BEFORE parsing.
        
        This helps identify potential issues early!
        
        Returns:
            Dict with analysis results:
            - complexity: How complex is the instruction?
            - ambiguity: Any unclear parts?
            - suggested_actions: How many actions we expect
        """
        instruction_lower = instruction.lower()
        
        # Count action keywords
        action_keywords = ['pick', 'place', 'put', 'move', 'go', 'open', 
                          'close', 'push', 'pull', 'grab', 'take', 'get']
        
        action_count = sum(1 for keyword in action_keywords 
                          if keyword in instruction_lower)
        
        # Check for connecting words (indicate multi-step)
        connectors = ['and', 'then', 'after', 'before']
        has_connectors = any(conn in instruction_lower for conn in connectors)
        
        # Determine complexity
        if action_count == 0:
            complexity = "unclear"
            ambiguity = "high"
        elif action_count == 1 and not has_connectors:
            complexity = "simple"
            ambiguity = "low"
        elif action_count <= 2 or has_connectors:
            complexity = "moderate"
            ambiguity = "low"
        else:
            complexity = "complex"
            ambiguity = "medium"
        
        return {
            "complexity": complexity,
            "ambiguity": ambiguity,
            "suggested_actions": max(1, action_count),
            "has_connectors": has_connectors,
            "word_count": len(instruction.split())
        }
    
    def parse_with_analysis(self, instruction: str) -> tuple:
        """
        Parse an instruction AND provide analysis.
        
        Returns:
            tuple: (plan, analysis_results)
        """
        # Analyze first
        analysis = self.analyze_instruction(instruction)
        
        # Warn about high ambiguity
        if analysis["ambiguity"] == "high":
            print("⚠️  Warning: Instruction may be unclear or ambiguous")
        
        # Parse
        plan = self.parse(instruction)
        
        # Add confidence score
        actual_actions = len(plan.actions)
        expected_actions = analysis["suggested_actions"]
        
        # Confidence based on how well we matched expectations
        if actual_actions == expected_actions:
            analysis["confidence"] = "high"
        elif abs(actual_actions - expected_actions) == 1:
            analysis["confidence"] = "medium"
        else:
            analysis["confidence"] = "low"
        
        analysis["actual_actions"] = actual_actions
        
        return plan, analysis


# Create advanced parser
advanced_parser = AdvancedRobotParser()

print("✅ Advanced parser created!")
print("\n🎯 New Features:")
print("   - Complexity analysis")
print("   - Ambiguity detection")
print("   - Confidence scoring")

## Step 11: Test Advanced Features

Let's test the advanced analysis capabilities with different types of instructions.

In [ ]:
# ========================================
# TEST ADVANCED FEATURES
# ========================================

test_cases = [
    "Pick up the mug",  # Simple
    "Go to the kitchen and open the fridge",  # Moderate
    "Grab the book, go to the desk, and place it next to the lamp",  # Complex
    "Do something with the thing"  # Unclear/ambiguous
]

print("🧪 Testing Advanced Features")
print("=" * 80)

for i, instruction in enumerate(test_cases, 1):
    print(f"\n[Test {i}] \"{instruction}\"")
    print("-" * 80)
    
    try:
        plan, analysis = advanced_parser.parse_with_analysis(instruction)
        
        print(f"\n📊 Analysis:")
        print(f"   Complexity: {analysis['complexity']}")
        print(f"   Ambiguity: {analysis['ambiguity']}")
        print(f"   Confidence: {analysis['confidence']}")
        print(f"   Expected actions: {analysis['suggested_actions']}")
        print(f"   Actual actions: {analysis['actual_actions']}")
        
        print(f"\n🤖 Commands:")
        for j, cmd in enumerate(plan.to_robot_commands(), 1):
            print(f"   {j}. {cmd}")
            
    except Exception as e:
        print(f"❌ Error: {e}")

print("\n" + "=" * 80)
print("✅ Advanced testing complete!")

## Step 12: Summary and Next Steps

🎉 **Congratulations!** You've built a complete natural language parser for robotic actions!

### What You've Learned:

1. **Grammar Design** - How to define structured robot commands
2. **LLM Integration** - Using AI to parse natural language
3. **Validation** - Ensuring plans are safe before execution
4. **Testing** - Comprehensive testing with diverse instructions
5. **Advanced Features** - Analysis and confidence scoring

### Key Takeaways:

- 🤖 **Robots need structure** - Convert flexible language to precise commands
- ✅ **Validation is crucial** - Always check plans before executing
- 📝 **Grammar enforcement** - Define clear rules for valid actions
- 🧪 **Test thoroughly** - Use diverse examples to ensure robustness

### Next Steps - How to Improve:

1. **Add More Actions** - Extend ActionType with more robot capabilities
2. **Better Validation** - Add physics checks (can robot reach? Is object too heavy?)
3. **Context Awareness** - Remember where objects are, what robot is holding
4. **Error Recovery** - Suggest fixes when parsing fails
5. **Real Robot Integration** - Connect to actual robot hardware!
6. **Visual Feedback** - Show robot simulation while planning
7. **Voice Input** - Accept spoken commands
8. **Multi-language** - Support languages beyond English

### Resources for Learning More:

- **Guidance Library**: https://github.com/guidance-ai/guidance
- **Research Papers**:
  - "Do As I Can, Not As I Say" (2022): https://arxiv.org/abs/2204.01691
  - "Code as Policies" (2022): https://arxiv.org/abs/2207.05608
- **Robotics Basics**: ROS (Robot Operating System) tutorials
- **LLM Fundamentals**: OpenAI documentation, Hugging Face courses

### Your Turn!

Try modifying the code:
1. Add a new action type (e.g., "rotate", "scan", "measure")
2. Create your own test instructions
3. Improve the validation logic
4. Add visualization of the plan

**Remember**: Every expert was once a beginner. Keep experimenting, and don't be afraid to make mistakes - that's how we learn! 🚀

## Bonus: Export Your Results

Save your test results to files so you can review them later!

In [ ]:
# ========================================
# EXPORT RESULTS
# ========================================

import json
from datetime import datetime

# Create export data
export_data = {
    "timestamp": datetime.now().isoformat(),
    "total_tests": len(results),
    "successful": successful_parses,
    "failed": failed_parses,
    "success_rate": f"{(successful_parses/len(results)*100):.1f}%",
    "results": results
}

# Save to JSON file
output_file = "robot_parser_results.json"
with open(output_file, "w") as f:
    json.dump(export_data, f, indent=2)

print(f"✅ Results exported to: {output_file}")

# Also save the DataFrame as CSV
csv_file = "robot_parser_results.csv"
df.to_csv(csv_file, index=False)
print(f"✅ Table exported to: {csv_file}")

print("\n📦 Export complete! You can now:")
print("   - Review results offline")
print("   - Share with others")
print("   - Compare different runs")

## 🎓 Final Challenge

Ready to test your understanding? Try solving these challenges:

### Challenge 1: Add a New Action Type
Add a "rotate" action that can rotate objects or the robot itself.

**Hint**: You'll need to:
1. Add `ROTATE = "rotate"` to the `ActionType` enum
2. Update the system prompt in `_create_system_prompt()`
3. Test it with instructions like "Rotate the chair 90 degrees"

### Challenge 2: Improve Validation
Add a check that prevents the robot from placing objects on unstable surfaces.

**Hint**: Create a list of stable surfaces (table, desk, shelf) and unstable ones (paper, cloth).

### Challenge 3: Context Memory
Make the parser remember the last location it moved to, so "pick up the mug" knows where to look.

**Hint**: Add a `current_location` attribute to the parser class that gets updated with each `move_to` action.

### Challenge 4: Create Your Own
Think of a real-world task you'd want a robot to do, and test if the parser can handle it!

Good luck, and happy robot programming! 🤖✨

## Step 13: Mapping Language to Objects with Vision-Language Models (VLMs) 🎯

### What's the Problem?

So far, we've been using simple object names like "mug" or "table". But in the real world:
- There might be **multiple mugs** (which one do you mean?)
- Objects have **attributes** like color, size, material
- Humans say things like "the blue mug on the left" or "the small red cup"

**The Challenge**: How do we map natural language descriptions to specific object IDs?

### What is a Vision-Language Model (VLM)?

A **VLM** is an AI model that understands **both images and text**. Think of it as having:
- 👁️ **Eyes** - Can see and analyze images
- 🧠 **Brain** - Can understand language descriptions
- 🔗 **Connection** - Can match language to visual objects

**Examples of VLMs**:
- **CLIP** (by OpenAI) - Matches images with text descriptions
- **GPT-4V** - Powerful vision + language understanding
- **LLaVA** - Open-source vision-language model

### What We'll Build:

A **Language-to-Object Grounding System** that:
1. Takes a language description: "the blue mug"
2. Looks at a scene with multiple objects
3. Identifies which specific object is being referred to
4. Returns structured output with confidence scores

Let's get started! 🚀